# Follower POGEMA Baseline — Google Colab

Runs the **Follower** baseline (pure-Python, no FollowerLite) across the POGEMA `experiments/` folders, pulling code from **`tay805/mapf-deadlock`**.

**Before running:** set `Runtime > Change runtime type > GPU` (T4 is fine).

**Why a conda env?** Follower's stack is pinned to `torch 1.13.1 / sample-factory 2.1.1 / numpy<=1.23.1 / pandas<=1.4`, which only have wheels up to **Python 3.10**. Colab's native Python is 3.12, where those fail to build. So cell 1 installs conda (`condacolab`), and cell 4 builds a dedicated **`follower` env on Python 3.10**. The eval runs via `conda run -n follower …`, so the kernel's own Python (3.12) is irrelevant.

**Speed note:** GPU accelerates the policy net, but A* + env stepping stay CPU-bound and Colab free has ~2 vCPU. The win is offloading your machine, not raw speed.

## 1. Install Python 3.10 env (kernel will restart)

In [ ]:
# Installs a Python 3.10 conda base. The kernel RESTARTS automatically at the end
# of this cell — that is normal. After it restarts, run cell 2 onward.
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Verify env + GPU (run AFTER the restart)

In [ ]:
import condacolab; condacolab.check()   # confirms the conda base is active
!conda --version                        # needed to build the py3.10 env in cell 4
# The kernel's own Python can be 3.12 — we don't use it; the eval runs in the
# 'follower' conda env (py3.10) built in the deps cell.
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU'

## 3. Clone the project repo

Public repo, so no auth needed. Weights are committed (~59MB), so the clone brings them along.

To pull later changes without re-cloning: `!git -C /content/mapf-deadlock pull`

In [ ]:
%cd /content
![ -d mapf-deadlock ] || git clone https://github.com/tay805/mapf-deadlock.git
%cd /content/mapf-deadlock
!ls -lh model/follower/checkpoint_p0/*.pth baseline_eval.py
# Mount Google Drive so generated RESULTS persist across sessions. Code stays in
# fast /content (re-cloning is cheap; checkpoints are in the repo); only results
# go to Drive via baseline_eval.py --out=RESULTS.
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS = '/content/drive/MyDrive/mapf-deadlock-results'
os.makedirs(RESULTS, exist_ok=True)
print('Results will be saved to:', RESULTS)

## 4. Build the Python 3.10 `follower` env + install deps

Creates a conda env on 3.10 and installs torch 1.13.1 + the eval's deps into it. Takes a few minutes. Must end with `OK | py 3.10.x | torch 1.13.1+cu117 | cuda True`.

In [ ]:
# Follower's stack REQUIRES Python 3.10 (torch 1.13.1, sample-factory 2.1.1,
# numpy<=1.23.1, pandas<=1.4) — on Colab's native 3.12 these have no wheels and
# fail to build from source. So we build a dedicated conda env 'follower' on 3.10
# and run everything through `conda run -n follower`. The kernel's own Python
# doesn't matter. (Requires conda: cell 1 / condacolab must have run first.)
!conda --version || echo "!! conda missing — run cell 1 (condacolab) and let the kernel RESTART, then re-run this cell"
!apt-get -qq install -y build-essential >/dev/null
!conda create -y -n follower python=3.10 2>&1 | tail -2
# CUDA torch 1.13.1 (cu117 wheel exists for py3.10; runs on Colab T4).
!conda run -n follower pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
# Everything else (wheels exist on 3.10). sample-factory pulls its own deps.
!conda run -n follower pip install -q --prefer-binary \
  sample-factory==2.1.1 pogema==1.3.0 pogema-toolbox==0.1.0 \
  "numpy>=1.21.5,<=1.23.1" "pandas<=1.4" matplotlib seaborn tabulate \
  pyyaml dask==2024.8.0 distributed==2024.8.0 loguru cppimport pybind11==2.13.1
# Verify INSIDE the env — must print py 3.10.x and torch 1.13.1+cu117.
!conda run -n follower python -c "import sys, torch, pogema, sample_factory, cppimport, pogema_toolbox, dask, loguru; print('OK | py', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

## 5. Quick validation run (warehouse, 1 seed — a few minutes)

`baseline_eval.py` lives in the repo (Follower-only, no wandb, per-folder JSON, `--seeds=N`). Confirms the stack before the long run.

In [ ]:
# --no-capture-output streams logs live. First run pauses ~30-60s to compile the
# C++ A* planner. Results -> Google Drive (RESULTS), so they survive disconnects.
!cd /content/mapf-deadlock && conda run --no-capture-output -n follower python baseline_eval.py --seeds=1 05-warehouse --out "/content/drive/MyDrive/mapf-deadlock-results"

## 6. Full 3-seed baseline pass (all folders)

Long-running. Keep the tab active so Colab doesn't disconnect. To split work, pass specific folders, e.g. `!python baseline_eval.py --seeds=3 02-mazes 05-warehouse`.

In [ ]:
# Full 3-seed pass, all folders. Long-running; keep the tab active.
# Results -> Google Drive, written per-folder so a disconnect keeps finished ones.
# To split work: append folders, e.g. ... baseline_eval.py --seeds=3 02-mazes --out ...
!cd /content/mapf-deadlock && conda run --no-capture-output -n follower python baseline_eval.py --seeds=3 --out "/content/drive/MyDrive/mapf-deadlock-results"

## 7. Results

Results are already persisted to Google Drive at `MyDrive/mapf-deadlock-results/` (per-folder `*.json` + plots), so they survive a disconnect. The cell below just lists them. To also archive into the repo, copy them into `experiments/` and `git push`.

In [ ]:
# Results already live on Drive (survive disconnects). List them:
!find /content/drive/MyDrive/mapf-deadlock-results -name '*.json' -o -name '*.pdf' | sort
# Optional: copy into the repo and push so they're versioned on GitHub too.
# !cp -r /content/drive/MyDrive/mapf-deadlock-results/* /content/mapf-deadlock/experiments/ \
#   && cd /content/mapf-deadlock && git add experiments && git commit -m "baseline results" && git push